In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

ROBUSTNESS_FIGURE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
    / "robustness_analysis"
)

ROBUSTNESS_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FACTOR_SCORE_FILE = (
    PROCESSED_DATA_DIR
    / "11_sp500_monthly_factor_scores_full_2005_2025.parquet"
)

BENCHMARK_FILE = (
    PROCESSED_DATA_DIR
    / "12_benchmark_returns_monthly_2005_2025.parquet"
)

print("Robustness-analysis environment was initialized successfully.")

In [ ]:
factor_score_df = pd.read_parquet(
    FACTOR_SCORE_FILE
)

benchmark_df = pd.read_parquet(
    BENCHMARK_FILE
)

factor_score_df["month"] = pd.to_datetime(
    factor_score_df["month"]
)

benchmark_df["month"] = pd.to_datetime(
    benchmark_df["month"]
)

factor_score_df = (
    factor_score_df
    .sort_values(
        ["month", "permno"]
    )
    .reset_index(drop=True)
)

benchmark_df = (
    benchmark_df
    .sort_values("month")
    .reset_index(drop=True)
)

print("Robustness-analysis datasets were loaded successfully.")
print("Factor-score rows:", len(factor_score_df))
print("Factor-score columns:", len(factor_score_df.columns))
print("Benchmark rows:", len(benchmark_df))

print("\nFactor-score date range:")
print(
    factor_score_df["month"].min().date(),
    "to",
    factor_score_df["month"].max().date()
)

print("\nBenchmark date range:")
print(
    benchmark_df["month"].min().date(),
    "to",
    benchmark_df["month"].max().date()
)

In [ ]:
required_factor_columns = [
    "permno",
    "month",
    "future_return_1m",
    "month_end_market_cap",
    "volatility_12m",
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",
    "multi_factor_score"
]

missing_factor_columns = [
    column
    for column in required_factor_columns
    if column not in factor_score_df.columns
]

if missing_factor_columns:
    print("Available factor-score columns:")
    for column in factor_score_df.columns:
        print(column)

    raise ValueError(
        f"Missing required columns: {missing_factor_columns}"
    )

duplicate_groups = (
    factor_score_df
    .duplicated(
        subset=["permno", "month"]
    )
    .sum()
)

if duplicate_groups != 0:
    raise ValueError(
        "Duplicate PERMNO-month observations were found."
    )

analysis_sample_df = (
    factor_score_df
    .loc[
        (
            factor_score_df["month"]
            >= pd.Timestamp("2014-12-31")
        )
        & (
            factor_score_df["month"]
            <= pd.Timestamp("2025-11-30")
        )
    ]
    .copy()
)

analysis_sample_df["return_month"] = (
    analysis_sample_df["month"]
    + pd.offsets.MonthEnd(1)
)

validation_summary = pd.DataFrame({
    "total_rows": [
        len(analysis_sample_df)
    ],

    "unique_months": [
        analysis_sample_df[
            "month"
        ].nunique()
    ],

    "unique_permnos": [
        analysis_sample_df[
            "permno"
        ].nunique()
    ],

    "missing_future_returns": [
        analysis_sample_df[
            "future_return_1m"
        ].isna().sum()
    ],

    "missing_multi_factor_scores": [
        analysis_sample_df[
            "multi_factor_score"
        ].isna().sum()
    ],

    "missing_quality_scores": [
        analysis_sample_df[
            "factor_quality"
        ].isna().sum()
    ],

    "missing_volatility": [
        analysis_sample_df[
            "volatility_12m"
        ].isna().sum()
    ]
})

print("Robustness-analysis sample validation:")
display(validation_summary)

print(
    "\nFormation-month range:",
    analysis_sample_df["month"].min().date(),
    "to",
    analysis_sample_df["month"].max().date()
)

print(
    "Return-month range:",
    analysis_sample_df["return_month"].min().date(),
    "to",
    analysis_sample_df["return_month"].max().date()
)

In [ ]:
signal_specifications = {
    "Quality":
        "factor_quality",

    "Six-Factor":
        "multi_factor_score"
}

selection_fraction_specifications = {
    "Top 10%": 0.10,
    "Top 20%": 0.20,
    "Top 30%": 0.30
}

weighting_methods = [
    "Equal Weight",
    "Score Rank Weight",
    "Inverse Volatility"
]

transaction_cost_specifications = {
    "0 bps": 0.0000,
    "10 bps": 0.0010,
    "25 bps": 0.0025,
    "50 bps": 0.0050
}

print("Robustness specifications:")
print(
    "Number of signals:",
    len(signal_specifications)
)
print(
    "Number of selection fractions:",
    len(selection_fraction_specifications)
)
print(
    "Number of weighting methods:",
    len(weighting_methods)
)
print(
    "Number of transaction-cost assumptions:",
    len(transaction_cost_specifications)
)

print(
    "Total strategy-cost combinations:",
    len(signal_specifications)
    * len(selection_fraction_specifications)
    * len(weighting_methods)
    * len(transaction_cost_specifications)
)

In [ ]:
def calculate_target_weights(
    selected_sample,
    signal_column,
    weighting_method
):
    weighting_sample = (
        selected_sample
        .copy()
    )

    if weighting_method == "Equal Weight":
        raw_weights = pd.Series(
            1.0,
            index=weighting_sample.index
        )

    elif weighting_method == "Score Rank Weight":
        raw_weights = (
            weighting_sample[
                signal_column
            ]
            .rank(
                method="average",
                pct=True
            )
        )

    elif weighting_method == "Inverse Volatility":
        weighting_sample = (
            weighting_sample
            .loc[
                weighting_sample[
                    "volatility_12m"
                ].notna()
                & (
                    weighting_sample[
                        "volatility_12m"
                    ]
                    > 0
                )
            ]
            .copy()
        )

        inverse_volatility = (
            1.0
            / weighting_sample[
                "volatility_12m"
            ]
        )

        lower_bound = (
            inverse_volatility.quantile(0.05)
        )

        upper_bound = (
            inverse_volatility.quantile(0.95)
        )

        raw_weights = (
            inverse_volatility
            .clip(
                lower=lower_bound,
                upper=upper_bound
            )
        )

    else:
        raise ValueError(
            f"Unknown weighting method: {weighting_method}"
        )

    if len(weighting_sample) == 0:
        raise ValueError(
            "No securities remained after weighting filters."
        )

    target_weight_sum = raw_weights.sum()

    if (
        not np.isfinite(target_weight_sum)
        or target_weight_sum <= 0
    ):
        raise ValueError(
            "Invalid target-weight denominator."
        )

    weighting_sample["target_weight"] = (
        raw_weights
        / target_weight_sum
    )

    return weighting_sample


target_weight_frames = []

for month, month_sample in (
    analysis_sample_df
    .groupby(
        "month",
        sort=True
    )
):
    for signal_name, signal_column in (
        signal_specifications.items()
    ):
        eligible_sample = (
            month_sample
            .loc[
                month_sample[
                    signal_column
                ].notna()
            ]
            .copy()
        )

        if len(eligible_sample) < 100:
            raise ValueError(
                f"Insufficient securities in {month} "
                f"for {signal_name}."
            )

        eligible_sample[
            "signal_percentile_rank"
        ] = (
            eligible_sample[
                signal_column
            ]
            .rank(
                method="first",
                pct=True
            )
        )

        for selection_label, selection_fraction in (
            selection_fraction_specifications.items()
        ):
            selected_sample = (
                eligible_sample
                .loc[
                    eligible_sample[
                        "signal_percentile_rank"
                    ]
                    > (
                        1.0
                        - selection_fraction
                    )
                ]
                .copy()
            )

            for weighting_method in weighting_methods:
                weighted_sample = (
                    calculate_target_weights(
                        selected_sample=selected_sample,
                        signal_column=signal_column,
                        weighting_method=weighting_method
                    )
                )

                strategy_name = (
                    f"{signal_name} | "
                    f"{selection_label} | "
                    f"{weighting_method}"
                )

                weighted_sample["strategy"] = (
                    strategy_name
                )

                weighted_sample["signal_name"] = (
                    signal_name
                )

                weighted_sample["selection_label"] = (
                    selection_label
                )

                weighted_sample[
                    "selection_fraction"
                ] = selection_fraction

                weighted_sample[
                    "weighting_method"
                ] = weighting_method

                target_weight_frames.append(
                    weighted_sample[
                        [
                            "strategy",
                            "signal_name",
                            "selection_label",
                            "selection_fraction",
                            "weighting_method",
                            "month",
                            "return_month",
                            "permno",
                            "ticker",
                            "target_weight",
                            "future_return_1m",
                            "month_end_market_cap",
                            "volatility_12m",
                            signal_column
                        ]
                    ].copy()
                )

robustness_target_weights_df = pd.concat(
    target_weight_frames,
    ignore_index=True
)

print("Robustness target weights were created successfully.")
print(
    "Number of target-weight records:",
    len(robustness_target_weights_df)
)
print(
    "Number of strategies:",
    robustness_target_weights_df[
        "strategy"
    ].nunique()
)

In [ ]:
weight_sum_validation_df = (
    robustness_target_weights_df
    .groupby(
        ["strategy", "month"],
        as_index=False
    )
    .agg(
        weight_sum=(
            "target_weight",
            "sum"
        ),

        number_of_holdings=(
            "permno",
            "nunique"
        ),

        missing_future_returns=(
            "future_return_1m",
            lambda values: values.isna().sum()
        )
    )
)

maximum_weight_sum_error = (
    weight_sum_validation_df[
        "weight_sum"
    ]
    .sub(1.0)
    .abs()
    .max()
)

duplicate_target_count = (
    robustness_target_weights_df
    .duplicated(
        subset=[
            "strategy",
            "month",
            "permno"
        ]
    )
    .sum()
)

if maximum_weight_sum_error > 1e-10:
    raise ValueError(
        "Target weights do not sum to one."
    )

if duplicate_target_count != 0:
    raise ValueError(
        "Duplicate strategy-month-PERMNO holdings were found."
    )

holdings_summary_df = (
    weight_sum_validation_df
    .groupby("strategy")
    .agg(
        minimum_holdings=(
            "number_of_holdings",
            "min"
        ),

        average_holdings=(
            "number_of_holdings",
            "mean"
        ),

        maximum_holdings=(
            "number_of_holdings",
            "max"
        ),

        total_missing_future_returns=(
            "missing_future_returns",
            "sum"
        )
    )
    .sort_index()
)

print("Target-weight validation:")
print(
    "Maximum absolute weight-sum error:",
    f"{maximum_weight_sum_error:.12f}"
)
print(
    "Duplicate target holdings:",
    duplicate_target_count
)

print("\nHoldings summary:")
display(
    holdings_summary_df.round(2)
)

In [ ]:
def run_drifted_weight_backtest(
    target_weight_data
):
    monthly_records = []

    for strategy_name, strategy_sample in (
        target_weight_data
        .groupby(
            "strategy",
            sort=True
        )
    ):
        strategy_sample = (
            strategy_sample
            .sort_values(
                ["month", "permno"]
            )
        )

        previous_drifted_weights = None

        for month, month_sample in (
            strategy_sample
            .groupby(
                "month",
                sort=True
            )
        ):
            month_sample = (
                month_sample
                .copy()
            )

            target_weights = (
                month_sample
                .set_index("permno")[
                    "target_weight"
                ]
                .astype(float)
            )

            realized_returns = (
                month_sample
                .set_index("permno")[
                    "future_return_1m"
                ]
                .astype(float)
            )

            missing_return_count = int(
                realized_returns
                .isna()
                .sum()
            )

            realized_returns_filled = (
                realized_returns
                .fillna(0.0)
            )

            if previous_drifted_weights is None:
                turnover = 1.0

            else:
                combined_permnos = (
                    target_weights.index
                    .union(
                        previous_drifted_weights.index
                    )
                )

                current_aligned_weights = (
                    target_weights
                    .reindex(
                        combined_permnos,
                        fill_value=0.0
                    )
                )

                previous_aligned_weights = (
                    previous_drifted_weights
                    .reindex(
                        combined_permnos,
                        fill_value=0.0
                    )
                )

                turnover = (
                    0.5
                    * (
                        current_aligned_weights
                        - previous_aligned_weights
                    )
                    .abs()
                    .sum()
                )

            gross_return = (
                target_weights
                * realized_returns_filled
            ).sum()

            end_of_month_values = (
                target_weights
                * (
                    1.0
                    + realized_returns_filled
                )
            )

            total_end_of_month_value = (
                end_of_month_values.sum()
            )

            if total_end_of_month_value <= 0:
                raise ValueError(
                    "Invalid end-of-month portfolio value."
                )

            previous_drifted_weights = (
                end_of_month_values
                / total_end_of_month_value
            )

            first_row = month_sample.iloc[0]

            monthly_records.append({
                "strategy":
                    strategy_name,

                "signal_name":
                    first_row["signal_name"],

                "selection_label":
                    first_row["selection_label"],

                "selection_fraction":
                    first_row["selection_fraction"],

                "weighting_method":
                    first_row["weighting_method"],

                "formation_month":
                    month,

                "return_month":
                    first_row["return_month"],

                "number_of_holdings":
                    len(target_weights),

                "missing_return_holdings":
                    missing_return_count,

                "turnover":
                    turnover,

                "gross_return":
                    gross_return
            })

    return pd.DataFrame(
        monthly_records
    )


robustness_gross_backtest_df = (
    run_drifted_weight_backtest(
        robustness_target_weights_df
    )
)

print("Gross robustness backtests were completed successfully.")
print(
    "Number of monthly strategy records:",
    len(robustness_gross_backtest_df)
)
print(
    "Number of strategies:",
    robustness_gross_backtest_df[
        "strategy"
    ].nunique()
)
print(
    "Months per strategy:",
    robustness_gross_backtest_df
    .groupby("strategy")
    .size()
    .min(),
    "to",
    robustness_gross_backtest_df
    .groupby("strategy")
    .size()
    .max()
)

In [ ]:
cost_adjusted_frames = []

for cost_label, cost_rate in (
    transaction_cost_specifications.items()
):
    cost_sample = (
        robustness_gross_backtest_df
        .copy()
    )

    cost_sample["cost_label"] = (
        cost_label
    )

    cost_sample["cost_rate"] = (
        cost_rate
    )

    cost_sample["transaction_cost"] = (
        cost_sample["turnover"]
        * cost_rate
    )

    cost_sample["net_return"] = (
        (
            1.0
            - cost_sample[
                "transaction_cost"
            ]
        )
        * (
            1.0
            + cost_sample[
                "gross_return"
            ]
        )
        - 1.0
    )

    cost_adjusted_frames.append(
        cost_sample
    )

robustness_backtest_df = pd.concat(
    cost_adjusted_frames,
    ignore_index=True
)

expected_combinations = 72

actual_combinations = (
    robustness_backtest_df[
        [
            "strategy",
            "cost_label"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

if actual_combinations != expected_combinations:
    raise ValueError(
        "The number of strategy-cost combinations is incorrect."
    )

print("Transaction-cost scenarios were applied successfully.")
print(
    "Number of strategy-cost combinations:",
    actual_combinations
)
print(
    "Total monthly observations:",
    len(robustness_backtest_df)
)

In [ ]:
FF_FACTOR_FILE = (
    PROCESSED_DATA_DIR
    / "15_fama_french_monthly_2015_2025.parquet"
)

ff_factor_df = pd.read_parquet(
    FF_FACTOR_FILE
)

ff_factor_df["month"] = pd.to_datetime(
    ff_factor_df["month"]
)

benchmark_merge_df = (
    benchmark_df[
        [
            "month",
            "crsp_value_weighted_total_return",
            "crsp_equal_weighted_total_return"
        ]
    ]
    .rename(
        columns={
            "month":
                "return_month"
        }
    )
)

ff_merge_df = (
    ff_factor_df
    .rename(
        columns={
            "month":
                "return_month"
        }
    )
)

robustness_evaluation_df = (
    robustness_backtest_df
    .merge(
        benchmark_merge_df,
        on="return_month",
        how="left",
        validate="many_to_one"
    )
    .merge(
        ff_merge_df,
        on="return_month",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "strategy",
            "cost_rate",
            "return_month"
        ]
    )
    .reset_index(drop=True)
)

evaluation_required_columns = [
    "net_return",
    "rf",
    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "mom",
    "crsp_equal_weighted_total_return",
    "crsp_value_weighted_total_return"
]

missing_evaluation_values = (
    robustness_evaluation_df[
        evaluation_required_columns
    ]
    .isna()
    .sum()
)

if missing_evaluation_values.sum() != 0:
    raise ValueError(
        "Missing benchmark or factor observations were found."
    )

print("Benchmark and factor data were merged successfully.")
print(
    "Number of monthly observations:",
    len(robustness_evaluation_df)
)
print("\nMissing values:")
print(missing_evaluation_values)

In [ ]:
def calculate_robustness_performance(
    group
):
    group = (
        group
        .sort_values("return_month")
        .copy()
    )

    returns = (
        group["net_return"]
        .astype(float)
    )

    risk_free_rate = (
        group["rf"]
        .astype(float)
    )

    excess_returns = (
        returns
        - risk_free_rate
    )

    benchmark_returns = (
        group[
            "crsp_equal_weighted_total_return"
        ]
        .astype(float)
    )

    active_returns = (
        returns
        - benchmark_returns
    )

    number_of_months = len(returns)

    terminal_wealth = (
        1.0 + returns
    ).prod()

    annualized_return = (
        terminal_wealth
        ** (
            12.0
            / number_of_months
        )
        - 1.0
    )

    annualized_volatility = (
        returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    annualized_sharpe = (
        excess_returns.mean()
        / excess_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    downside_excess_returns = np.minimum(
        excess_returns,
        0.0
    )

    annualized_downside_deviation = (
        np.sqrt(
            np.mean(
                downside_excess_returns ** 2
            )
        )
        * np.sqrt(12.0)
    )

    annualized_sortino = (
        excess_returns.mean()
        * 12.0
        / annualized_downside_deviation
        if annualized_downside_deviation > 0
        else np.nan
    )

    wealth = (
        1.0 + returns
    ).cumprod()

    drawdown = (
        wealth
        / wealth.cummax()
        - 1.0
    )

    maximum_drawdown = drawdown.min()

    fifth_percentile = (
        returns.quantile(0.05)
    )

    historical_cvar_95 = (
        -returns[
            returns <= fifth_percentile
        ].mean()
    )

    annualized_active_return = (
        active_returns.mean()
        * 12.0
    )

    annualized_tracking_error = (
        active_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    information_ratio = (
        annualized_active_return
        / annualized_tracking_error
        if annualized_tracking_error > 0
        else np.nan
    )

    return pd.Series({
        "number_of_months":
            number_of_months,

        "average_holdings":
            group[
                "number_of_holdings"
            ].mean(),

        "average_turnover":
            group[
                "turnover"
            ].mean(),

        "annualized_return":
            annualized_return,

        "annualized_volatility":
            annualized_volatility,

        "annualized_sharpe":
            annualized_sharpe,

        "annualized_sortino":
            annualized_sortino,

        "annualized_downside_deviation":
            annualized_downside_deviation,

        "maximum_drawdown":
            maximum_drawdown,

        "historical_cvar_95":
            historical_cvar_95,

        "annualized_active_return":
            annualized_active_return,

        "annualized_tracking_error":
            annualized_tracking_error,

        "information_ratio":
            information_ratio,

        "positive_month_rate":
            (returns > 0).mean(),

        "terminal_wealth":
            terminal_wealth,

        "total_missing_return_holdings":
            group[
                "missing_return_holdings"
            ].sum()
    })


robustness_performance_df = (
    robustness_evaluation_df
    .groupby(
        [
            "strategy",
            "signal_name",
            "selection_label",
            "selection_fraction",
            "weighting_method",
            "cost_label",
            "cost_rate"
        ],
        sort=True
    )
    .apply(
        calculate_robustness_performance
    )
    .reset_index()
)

if len(robustness_performance_df) != 72:
    raise ValueError(
        "The robustness performance table should contain 72 rows."
    )

print("Robustness performance metrics were calculated successfully.")
print(
    "Number of strategy-cost results:",
    len(robustness_performance_df)
)

In [ ]:
primary_cost_performance_df = (
    robustness_performance_df
    .loc[
        robustness_performance_df[
            "cost_label"
        ]
        == "10 bps"
    ]
    .copy()
)

primary_display_columns = [
    "strategy",
    "average_holdings",
    "average_turnover",
    "annualized_return",
    "annualized_volatility",
    "annualized_sharpe",
    "annualized_sortino",
    "maximum_drawdown",
    "historical_cvar_95",
    "information_ratio"
]

primary_performance_ranking_df = (
    primary_cost_performance_df[
        primary_display_columns
    ]
    .sort_values(
        "annualized_sharpe",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Strategy ranking under a 10-bps transaction cost:")
display(
    primary_performance_ranking_df.round(4)
)

In [ ]:
for signal_name in [
    "Quality",
    "Six-Factor"
]:
    signal_sample = (
        primary_cost_performance_df
        .loc[
            primary_cost_performance_df[
                "signal_name"
            ]
            == signal_name
        ]
    )

    sharpe_pivot = (
        signal_sample
        .pivot(
            index="selection_label",
            columns="weighting_method",
            values="annualized_sharpe"
        )
        .reindex(
            [
                "Top 10%",
                "Top 20%",
                "Top 30%"
            ]
        )
    )

    return_pivot = (
        signal_sample
        .pivot(
            index="selection_label",
            columns="weighting_method",
            values="annualized_return"
        )
        .reindex(
            [
                "Top 10%",
                "Top 20%",
                "Top 30%"
            ]
        )
    )

    drawdown_pivot = (
        signal_sample
        .pivot(
            index="selection_label",
            columns="weighting_method",
            values="maximum_drawdown"
        )
        .reindex(
            [
                "Top 10%",
                "Top 20%",
                "Top 30%"
            ]
        )
    )

    print(
        f"\n{signal_name} annualized Sharpe ratios:"
    )
    display(sharpe_pivot.round(4))

    print(
        f"{signal_name} annualized returns:"
    )
    display(return_pivot.round(4))

    print(
        f"{signal_name} maximum drawdowns:"
    )
    display(drawdown_pivot.round(4))

In [ ]:
cost_return_pivot_df = (
    robustness_performance_df
    .pivot(
        index="strategy",
        columns="cost_label",
        values="annualized_return"
    )
    .reindex(
        columns=[
            "0 bps",
            "10 bps",
            "25 bps",
            "50 bps"
        ]
    )
)

cost_return_pivot_df[
    "return_reduction_0_to_50_bps"
] = (
    cost_return_pivot_df["0 bps"]
    - cost_return_pivot_df["50 bps"]
)

cost_sharpe_pivot_df = (
    robustness_performance_df
    .pivot(
        index="strategy",
        columns="cost_label",
        values="annualized_sharpe"
    )
    .reindex(
        columns=[
            "0 bps",
            "10 bps",
            "25 bps",
            "50 bps"
        ]
    )
)

cost_sensitivity_summary_df = (
    cost_return_pivot_df
    .join(
        cost_sharpe_pivot_df,
        lsuffix="_annualized_return",
        rsuffix="_annualized_sharpe"
    )
    .sort_values(
        "return_reduction_0_to_50_bps",
        ascending=False
    )
)

print("Transaction-cost sensitivity:")
display(
    cost_sensitivity_summary_df.round(4)
)

In [ ]:
import statsmodels.api as sm

robustness_factor_columns = [
    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "mom"
]

primary_cost_monthly_df = (
    robustness_evaluation_df
    .loc[
        robustness_evaluation_df[
            "cost_label"
        ]
        == "10 bps"
    ]
    .copy()
)

robustness_alpha_records = []

for strategy_name, strategy_sample in (
    primary_cost_monthly_df
    .groupby(
        "strategy",
        sort=True
    )
):
    strategy_sample = (
        strategy_sample
        .dropna(
            subset=[
                "net_return",
                "rf"
            ]
            + robustness_factor_columns
        )
        .sort_values("return_month")
    )

    excess_return = (
        strategy_sample["net_return"]
        - strategy_sample["rf"]
    )

    independent_variables = sm.add_constant(
        strategy_sample[
            robustness_factor_columns
        ],
        has_constant="add"
    )

    model = sm.OLS(
        excess_return,
        independent_variables
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    first_row = strategy_sample.iloc[0]

    robustness_alpha_records.append({
        "strategy":
            strategy_name,

        "signal_name":
            first_row["signal_name"],

        "selection_label":
            first_row["selection_label"],

        "weighting_method":
            first_row["weighting_method"],

        "number_of_months":
            int(model.nobs),

        "annualized_alpha":
            model.params["const"] * 12.0,

        "alpha_t_statistic":
            model.tvalues["const"],

        "alpha_p_value":
            model.pvalues["const"],

        "mkt_rf_beta":
            model.params["mkt_rf"],

        "smb_beta":
            model.params["smb"],

        "hml_beta":
            model.params["hml"],

        "rmw_beta":
            model.params["rmw"],

        "cma_beta":
            model.params["cma"],

        "mom_beta":
            model.params["mom"],

        "adjusted_r_squared":
            model.rsquared_adj
    })

robustness_alpha_df = (
    pd.DataFrame(
        robustness_alpha_records
    )
    .sort_values(
        "annualized_alpha",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Factor-adjusted robustness results:")
display(
    robustness_alpha_df[
        [
            "strategy",
            "annualized_alpha",
            "alpha_t_statistic",
            "alpha_p_value",
            "mkt_rf_beta",
            "smb_beta",
            "hml_beta",
            "rmw_beta",
            "cma_beta",
            "mom_beta",
            "adjusted_r_squared"
        ]
    ].round(4)
)

In [ ]:
ROBUSTNESS_TARGET_FILE = (
    PROCESSED_DATA_DIR
    / "33_robustness_target_weights.parquet"
)

ROBUSTNESS_MONTHLY_FILE = (
    PROCESSED_DATA_DIR
    / "34_robustness_monthly_returns.csv"
)

ROBUSTNESS_PERFORMANCE_FILE = (
    PROCESSED_DATA_DIR
    / "35_robustness_performance_summary.csv"
)

ROBUSTNESS_ALPHA_FILE = (
    PROCESSED_DATA_DIR
    / "36_robustness_factor_alpha.csv"
)

COST_SENSITIVITY_FILE = (
    PROCESSED_DATA_DIR
    / "37_transaction_cost_sensitivity.csv"
)

robustness_target_weights_df.to_parquet(
    ROBUSTNESS_TARGET_FILE,
    index=False
)

robustness_evaluation_df.to_csv(
    ROBUSTNESS_MONTHLY_FILE,
    index=False
)

robustness_performance_df.to_csv(
    ROBUSTNESS_PERFORMANCE_FILE,
    index=False
)

robustness_alpha_df.to_csv(
    ROBUSTNESS_ALPHA_FILE,
    index=False
)

cost_sensitivity_summary_df.to_csv(
    COST_SENSITIVITY_FILE
)

print("Robustness-analysis files were saved successfully.")